# Парсинг паркингов Алматы из 2ГИС (Selenium, без API)

Ноутбук собирает список паркингов/автостоянок в Алматы с сайта `2gis.kz`, используя только Selenium (без официального API 2ГИС).

**Что делает скрипт:**

1. Открывает поиск 2ГИС по нескольким запросам (`паркинг`, `автостоянка`, `платная парковка` и т.д.).
2. Скроллит выдачу и собирает ссылки на карточки заведений — в самой ссылке уже зашиты координаты и id.
3. Открывает каждую карточку и забирает название и весь видимый текст страницы.
4. Из текста регулярками вытаскивает: адрес, часы работы, платность/тариф, вместимость, категорию.
5. Эвристически классифицирует тип объекта (ТЦ/БЦ/ЖК/городская/частная и т.д.) и пытается определить, к какому объекту относится паркинг (по названию).
6. Чистит дубликаты и нерелевантные результаты, сохраняет в CSV и JSON.

**Поля в итоговой таблице:**

- название
- адрес
- широта / долгота
- ссылка на карточку в 2ГИС
- платность (платная / бесплатная / неизвестно)
- тариф (если найден в тексте карточки)
- количество мест (если указано)
- тип (ТЦ/ТРЦ, БЦ, ЖК, городская, гостиница, аэропорт/вокзал, медцентр, частная/не определена)
- к какому объекту относится (эвристика по названию)
- категория из 2ГИС (как она встречается в тексте карточки)
- часы работы
- id карточки в 2ГИС

**Важные замечания:**

- 2ГИС часто меняет вёрстку и использует обфусцированные CSS-классы, поэтому большинство полей собирается через regex по полному тексту страницы — это устойчивее к изменениям вёрстки, но не идеально. Полный текст каждой карточки сохраняется в `raw_text` (файл `parking_almaty_full.json`), чтобы можно было дотянуть недостающие данные руками или улучшить регулярки.
- На первом запуске рекомендуется `HEADLESS = False`, чтобы видеть, не возникает ли капча/блокировка.
- Между запросами стоят случайные паузы — не убирайте их, иначе можно быстро попасть под анти-бот защиту 2ГИС.
- Сбор резюмируемый: уже обработанные карточки сохраняются в `parking_almaty_raw.jsonl`, при повторном запуске они не скачиваются заново.


In [1]:
!pip install -q selenium pandas webdriver-manager openpyxl


In [2]:
import os
import json
import re
import time
import random
from urllib.parse import quote

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


In [3]:
CITY = "almaty"
BASE_URL = "https://2gis.kz"

# Запросы для покрытия разных типов паркингов.
# Можно дополнять (например, добавить названия районов/ТРЦ Алматы),
# чтобы получить больше уникальных карточек.
SEARCH_QUERIES = [
    "паркинг",
    "парковка",
    "автостоянка",
    "платная парковка",
    "подземный паркинг",
    "крытый паркинг",
    "многоуровневый паркинг",
    "паркинг тц",
    "паркинг бц",
    "паркинг жк",
]

MAX_SCROLLS_PER_QUERY = 40   # сколько раз докручивать список результатов
SCROLL_PAUSE = 1.3           # базовая пауза между скроллами, сек

HEADLESS = False             # для первого прогона лучше False, чтобы видеть, что происходит

CHECKPOINT_FILE = "parking_almaty_raw.jsonl"
OUTPUT_CSV = "parking_almaty.csv"
OUTPUT_JSON_FULL = "parking_almaty_full.json"


In [4]:
def get_driver(headless=HEADLESS):
    options = Options()
    if headless:
        options.add_argument("--headless=new")

    options.add_argument("--lang=ru-RU")
    options.add_argument("--window-size=1400,1000")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options,
    )
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"},
    )
    return driver


driver = get_driver()


In [5]:
def close_popups(driver):
    """Пытаемся закрыть баннеры/попапы, которые могут перекрывать страницу."""
    xpaths = [
        "//button[contains(., 'Хорошо')]",
        "//button[contains(., 'Принять')]",
        "//button[contains(., 'Понятно')]",
        "//button[contains(., 'Закрыть')]",
        "//*[@aria-label='Close']",
    ]
    for xp in xpaths:
        try:
            for el in driver.find_elements(By.XPATH, xp):
                try:
                    el.click()
                    time.sleep(0.3)
                except Exception:
                    pass
        except Exception:
            pass


def parse_firm_id(url):
    m = re.search(r"/firm/(\d+)", url)
    return m.group(1) if m else None


def parse_coords_from_url(url):
    """Координаты часто зашиты прямо в ссылку на карточку: /firm/<id>/<lon>,<lat>"""
    m = re.search(r"/firm/\d+/([\d.]+)%2C([\d.]+)", url)
    if not m:
        m = re.search(r"/firm/\d+/([\d.]+),([\d.]+)", url)
    if m:
        lon, lat = float(m.group(1)), float(m.group(2))
        return lat, lon
    return None, None


def scroll_results_panel(driver):
    """Находим скроллируемую панель результатов поиска и скроллим её вниз."""
    driver.execute_script(
        """
        const anchors = document.querySelectorAll('a[href*="/firm/"]');
        if (!anchors.length) return;
        let el = anchors[anchors.length - 1];
        while (el && el.parentElement && el.scrollHeight <= el.clientHeight + 5) {
            el = el.parentElement;
        }
        if (el) {
            el.scrollTop = el.scrollHeight;
        }
        """
    )


In [6]:
def collect_firm_links(driver, query, max_scrolls=MAX_SCROLLS_PER_QUERY, pause=SCROLL_PAUSE):
    """Открывает поиск 2ГИС по запросу и собирает уникальные ссылки на карточки.

    Ссылка вида /almaty/firm/<id>/<lon>,<lat> уже содержит id заведения и координаты,
    поэтому для базового набора данных даже не обязательно открывать саму карточку.
    """
    url = f"{BASE_URL}/{CITY}/search/{quote(query)}"
    driver.get(url)
    time.sleep(3.5)
    close_popups(driver)

    links = {}
    last_count = 0
    stable_rounds = 0

    for step in range(max_scrolls):
        anchors = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/firm/"]')
        for a in anchors:
            href = a.get_attribute("href")
            if not href:
                continue
            fid = parse_firm_id(href)
            if fid and fid not in links:
                links[fid] = href

        if len(links) == last_count:
            stable_rounds += 1
        else:
            stable_rounds = 0
        last_count = len(links)

        if stable_rounds >= 3:
            break

        scroll_results_panel(driver)
        time.sleep(pause + random.uniform(0, 0.6))

    return links


In [7]:
def scrape_firm_detail(driver, href, timeout=12):
    """Открывает карточку заведения и забирает название + весь видимый текст страницы.

    Полный текст (raw_text) дальше разбирается регулярками: адрес, часы работы,
    платность/тариф, вместимость, категория.
    """
    driver.get(href)
    close_popups(driver)

    try:
        WebDriverWait(driver, timeout).until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
    except Exception:
        pass

    time.sleep(random.uniform(0.8, 1.6))

    data = {"firm_id": parse_firm_id(href), "link": href}

    lat, lon = parse_coords_from_url(href)
    data["lat"] = lat
    data["lon"] = lon

    try:
        data["name"] = driver.find_element(By.TAG_NAME, "h1").text.strip()
    except Exception:
        data["name"] = None

    try:
        data["raw_text"] = driver.find_element(By.TAG_NAME, "body").text
    except Exception:
        data["raw_text"] = ""

    return data


In [8]:
def extract_address(raw_text):
    """Адрес обычно находится в одной из первых строк карточки и содержит "Алматы"."""
    lines = [l.strip() for l in raw_text.splitlines() if l.strip()]
    for line in lines[:25]:
        if "Алматы" in line and len(line) < 120:
            return line
    return None


def extract_category(raw_text):
    """Категория/рубрика обычно встречается рядом с названием и содержит слово "парков"/"стоянк"."""
    lines = [l.strip() for l in raw_text.splitlines() if l.strip()]
    for line in lines[:25]:
        if re.search(r"парков|стоянк", line, re.IGNORECASE) and len(line) < 100:
            return line
    return None


_HOURS_PATTERNS = [
    r"(Круглосуточно)",
    r"(ежедневно[^\n]{0,60})",
    r"((?:Пн|Вт|Ср|Чт|Пт|Сб|Вс)[^\n]{0,80}\d{1,2}:\d{2}[^\n]{0,80})",
    r"(\d{1,2}:\d{2}\s*[-–]\s*\d{1,2}:\d{2})",
]


def extract_hours(raw_text):
    for pat in _HOURS_PATTERNS:
        m = re.search(pat, raw_text, re.IGNORECASE)
        if m:
            return m.group(1).strip()
    return None


def extract_pricing(raw_text):
    """Возвращает (платность, тариф). Платность: 'платная' / 'бесплатная' / None."""
    text_lower = raw_text.lower()
    is_free = bool(re.search(r"бесплатн", text_lower))
    is_paid = bool(re.search(r"платн(ая|ый|о)|тариф|₸|тенге", text_lower))

    tariff = None
    m = re.search(r"([\d\s]{1,6}\s?₸[^\n]{0,40})", raw_text)
    if not m:
        m = re.search(r"(\d{1,5}\s?тенге[^\n]{0,40})", raw_text, re.IGNORECASE)
    if m:
        tariff = re.sub(r"\s+", " ", m.group(1)).strip()

    if is_free and not is_paid:
        payment = "бесплатная"
    elif is_paid:
        payment = "платная"
    else:
        payment = None

    return payment, tariff


def extract_capacity(raw_text):
    """Ищем упоминание количества машино-мест."""
    m = re.search(
        r"(\d{1,5})\s*(?:машино-?\s?мест|парковочн\w*\s+мест\w*|мест\w*\s+для\s+парковки)",
        raw_text,
        re.IGNORECASE,
    )
    return m.group(1) if m else None


_TYPE_KEYWORDS = {
    "ТЦ/ТРЦ": ["трц", "тц ", "молл", "mall", "plaza", "mega", "dostyk"],
    "БЦ": ["бц", "бизнес-центр", "бизнес центр", "business center"],
    "ЖК": ["жк ", "жилой комплекс", "residence"],
    "Городская": ["городск", "муницип", "акимат", "общественн"],
    "Гостиница": ["отель", "hotel", "гостиниц"],
    "Аэропорт/вокзал": ["аэропорт", "вокзал", "airport"],
    "Медицинский центр": ["больниц", "клиник", "медицинск", "госпиталь"],
}


def classify_type(name, raw_text):
    """Эвристическая классификация типа объекта по ключевым словам в названии и тексте карточки."""
    text = f"{name or ''} {raw_text or ''}".lower()
    for type_name, keywords in _TYPE_KEYWORDS.items():
        if any(kw in text for kw in keywords):
            return type_name
    return "Частная / не определена"


def extract_parent_object(name):
    """Пытаемся понять, к какому объекту относится паркинг, по его названию.

    Например: 'Паркинг ТРЦ «Mega Park»' -> 'Mega Park'.
    Если кавычек нет, убираем общие слова (паркинг/парковка/...) и оставляем остаток.
    """
    if not name:
        return None
    m = re.search(r"[«\"']([^»\"']{2,60})[»\"']", name)
    if m:
        return m.group(1).strip()
    cleaned = re.sub(r"(?i)\b(паркинг|парковка|автостоянка|стоянка)\b", "", name)
    cleaned = cleaned.strip(" -–—,")
    return cleaned or None


In [9]:
all_links = {}

for q in SEARCH_QUERIES:
    print(f"Сбор ссылок по запросу: {q!r}")
    found = collect_firm_links(driver, q)
    print(f"  найдено уникальных карточек: {len(found)}")
    all_links.update(found)
    time.sleep(random.uniform(1.0, 2.0))

print(f"\nВсего уникальных карточек по всем запросам: {len(all_links)}")


Сбор ссылок по запросу: 'паркинг'
  найдено уникальных карточек: 0
Сбор ссылок по запросу: 'парковка'
  найдено уникальных карточек: 0
Сбор ссылок по запросу: 'автостоянка'
  найдено уникальных карточек: 0
Сбор ссылок по запросу: 'платная парковка'
  найдено уникальных карточек: 0
Сбор ссылок по запросу: 'подземный паркинг'
  найдено уникальных карточек: 0
Сбор ссылок по запросу: 'крытый паркинг'
  найдено уникальных карточек: 0
Сбор ссылок по запросу: 'многоуровневый паркинг'
  найдено уникальных карточек: 0
Сбор ссылок по запросу: 'паркинг тц'
  найдено уникальных карточек: 0
Сбор ссылок по запросу: 'паркинг бц'
  найдено уникальных карточек: 0
Сбор ссылок по запросу: 'паркинг жк'
  найдено уникальных карточек: 0

Всего уникальных карточек по всем запросам: 0


In [10]:
results = []
processed_ids = set()

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            results.append(rec)
            processed_ids.add(rec["firm_id"])
    print(f"Загружено {len(results)} ранее обработанных записей из {CHECKPOINT_FILE}")

items = list(all_links.items())

with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f:
    for i, (fid, href) in enumerate(items, start=1):
        if fid in processed_ids:
            continue
        try:
            data = scrape_firm_detail(driver, href)
            f.write(json.dumps(data, ensure_ascii=False) + "\n")
            f.flush()
            results.append(data)
            processed_ids.add(fid)
            print(f"[{i}/{len(items)}] OK: {data.get('name')}")
        except Exception as e:
            print(f"[{i}/{len(items)}] ОШИБКА для {href}: {e}")

        time.sleep(random.uniform(1.5, 3.0))

        # дополнительная пауза каждые 25 запросов, чтобы не попасть под анти-бот защиту
        if i % 25 == 0:
            time.sleep(random.uniform(5, 10))

print(f"\nВсего собрано записей: {len(results)}")



Всего собрано записей: 0


In [11]:
df = pd.DataFrame(results)

df["address"] = df["raw_text"].apply(extract_address)
df["category"] = df["raw_text"].apply(extract_category)
df["working_hours"] = df["raw_text"].apply(extract_hours)
df[["payment_type", "tariff"]] = df["raw_text"].apply(lambda t: pd.Series(extract_pricing(t)))
df["capacity"] = df["raw_text"].apply(extract_capacity)
df["type"] = df.apply(lambda r: classify_type(r["name"], r["raw_text"]), axis=1)
df["parent_object"] = df["name"].apply(extract_parent_object)

df.shape


KeyError: 'raw_text'

In [ ]:
# убираем явный мусор: записи без названия/координат
df = df.dropna(subset=["name", "lat", "lon"])
df = df[df["name"].str.len() > 0]

# оставляем только записи, где название или категория связаны с паркингом/стоянкой
mask_relevant = (
    df["name"].str.contains("парк|стоянк", case=False, na=False)
    | df["category"].fillna("").str.contains("парк|стоянк", case=False, na=False)
)
df = df[mask_relevant]

# дедуп по id карточки 2ГИС и по связке (название, адрес)
df = df.drop_duplicates(subset=["firm_id"])
df = df.drop_duplicates(subset=["name", "address"])

df = df.reset_index(drop=True)
print(f"Итого после очистки: {len(df)} записей")


In [ ]:
final_columns = [
    "name", "address", "lat", "lon", "link",
    "payment_type", "tariff", "capacity", "type",
    "parent_object", "category", "working_hours", "firm_id",
]

df_final = df[final_columns].rename(columns={
    "name": "название",
    "address": "адрес",
    "lat": "широта",
    "lon": "долгота",
    "link": "ссылка_2gis",
    "payment_type": "платность",
    "tariff": "тариф",
    "capacity": "количество_мест",
    "type": "тип",
    "parent_object": "относится_к_объекту",
    "category": "категория_2gis",
    "working_hours": "часы_работы",
    "firm_id": "id_2gis",
})

df_final.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
df.to_json(OUTPUT_JSON_FULL, orient="records", force_ascii=False, indent=2)

print(f"Сохранено {len(df_final)} строк в {OUTPUT_CSV}")
df_final.head(20)


In [ ]:
driver.quit()


## Возможные доработки

- Поле `raw_text` (сохраняется в `parking_almaty_full.json`) содержит весь видимый текст карточки 2ГИС. Если автоматические regex что-то не вытащили (часы работы, тариф, вместимость, точная категория) — посмотрите на конкретные примеры `raw_text` и доработайте регулярки в ячейке с функциями `extract_*`.
- Если 2ГИС начинает показывать капчу/заглушку "доступ ограничен":
  - увеличьте паузы (`SCROLL_PAUSE`, паузы между карточками в ячейке скрапинга деталей);
  - запускайте с `HEADLESS = False`;
  - запускайте сбор частями — скрипт умеет докачивать с того места, где остановился, благодаря `parking_almaty_raw.jsonl`.
- Классификация `тип` и `относится_к_объекту` — эвристическая (по ключевым словам и кавычкам в названии). Стоит вручную проверить записи с типом "Частная / не определена" — там часто прячутся ТЦ/ЖК/БЦ с нестандартными названиями.
- Список `SEARCH_QUERIES` можно расширить (например, добавить названия конкретных ТРЦ, БЦ, ЖК или районов Алматы), чтобы 2ГИС выдавал больше уникальных карточек и покрытие было полнее.
- Для дополнительной фотопроверки можно сохранять скриншот каждой карточки (`driver.save_screenshot(...)`) — полезно при отладке регулярок.


In [13]:
import requests
import pandas as pd
import time
import json

API_KEY = "c7f1a769-c8a5-4636-b14d-d8c987808a12"

TOKEN = "PASTE_CURRENT_X_USER_TOKEN_HERE"

URL = "https://catalog.api.2gis.ru/3.0/items"

headers = {
    "User-Agent": "Mozilla/5.0",
    "X-User-Token": "eyJhbGciOiJSU0EtT0FFUC0yNTYiLCJlbmMiOiJBMjU2R0NNIn0.NhcKHfnob5w_SQEGql1PC9VUP0Ui4iK7y8hz-F6X5V8_uVuB-AcLdUSqQBUlNbaVgFYWjMVPJ8jKoscZKgXmZgg56IU7vqwa969SljPhHGqNkLGqHa8elcUQzticeRs0NVmRTcjR2eFLnkSAU5BmOZui0wPZVQ4I1CnmEgt_JEYxPPoXiKmXBThT8nBB6ZgUKWLQ8rhDcI_QNAloL0kJgV6t0UX0jbMeFtE0ZhAGi0CRP7_mBtLukePhZV5AGENowXXo7Y1O-gDFvvQrrto820RK8m90SxgrRFQsA7RKiLb_-4svyG6jL50lOObx7FD2zqmYZ1aF_T65HF6a6IJE9w.VhuZcrEl1362APkl.JaefWfxo1onmDctLkpa6p4IdTefOV3AzkuBUxfhCctXYlpbxJ7PfkLQGe7W_VkutAIxhPkflYN1QFPb1gDQy2FEl74Ii_WWmBMqAYR44pIKk5IKXPCnTYUcrbK8wATuSG9PhybyABdWQwhUo_tIYAQK5xieaqfT3lOMFfs-JQvy4xs_Xg8CmwihcjbHPy-flHrXLr8EAa93zr8__5qcHBj95ktfHJcbcLNAJN0kCnNrsGIcOLqk_OUuQTRpdEqcrFCNEmEbWyuH2O9y79sCuoZGjzMHHH32rWDfaL0cUmmU8V9G2YmjKJFrPR2H9LKly5Uyle_LPkcPC-rlmoj1ndD_0pVe1jsy5y5jrTEiITsUqeIffLf8_aFl-nR9u0wDcgh_ofcC7RwilMVqPRRMemNBBfmVv8VQpuqKHnk5F-8Ebg2FRCh4oNVhOLjMsWGM-l0odKksW1KEp9jRopCXrI33-2NVaciPR7SThEgZwhVWipg8Yd7zq9BJBzjT1g9Avhi69-RXzVVsQsfga7qT3wMeTHiR-cnG6LA79uYGQgNEiBBM9RTG6mMJsJmye6g_eiuY9Pk0F4lMj6OxPKH3O5xGF9wld03trvEz71XpGWcbwsa53bBbSZz-5zvAkKPgCIxQRpDjH1O4azEWKAOkoTuPLz4A-SoF523grVVwYGs8z_11j31pd5f1aBD-3Y6XH7_2uaPkjrZA8j79isd3SjinrEAx7pcfPj0HAIa_I_wohVpQ8DeImxTD2gO_bTMSRHoBA6E15STj1H3Q9NrNNhGjb-VnChYhGmARFQt_1tcQSeUVbRN-Wj6r9TqWhzzNIkb2yTITHV1Wbudr3_3SVXOn1EXew5vBNKzvrqtQJGQaqFSlJpNFznNvvdip497RUpY8JYhpLVHk50y54v_Wx7HEeoVk86V4EvWjADGQMbBpFqCIAntWtkuyI3te5R73sK6Vt.2GkGrm0cvTOJXf-e-3Vv8Q",
    "Accept": "application/json, text/plain, */*",
    "Origin": "https://2gis.kz",
    "Referer": "https://2gis.kz/"
}

rows = []

page = 1

while True:

    params = {
        "key": API_KEY,
        "q": "парковки",
        "page": page,
        "page_size": 12,

        "locale": "ru_KZ",
        "allow_deleted": "true",
        "search_device_type": "desktop",

        "search_user_hash": "3486692531787608419",

        "viewpoint1": "76.756914,43.204700",
        "viewpoint2": "76.998420,43.192009"
    }

    try:

        r = requests.get(
            URL,
            params=params,
            headers=headers,
            timeout=30
        )

        print(f"\nPage {page}")
        print("Status:", r.status_code)

        try:
            data = r.json()
        except Exception:
            print("NOT JSON RESPONSE")
            print(r.text[:5000])
            break

        if "result" not in data:
            print("\nNO RESULT FOUND")
            print(json.dumps(data, indent=2, ensure_ascii=False)[:5000])
            break

        items = data["result"].get("items", [])

        if not items:
            print("No more items.")
            break

        print("Items:", len(items))

        for item in items:

            district = None

            for adm in item.get("adm_div", []):
                if adm.get("type") == "district":
                    district = adm.get("name")
                    break

            rubrics = "; ".join(
                r.get("name", "")
                for r in item.get("rubrics", [])
            )

            payment_methods = []

            for group in item.get("attribute_groups", []):
                if group.get("name") == "Способы оплаты":
                    payment_methods.extend(
                        attr.get("name", "")
                        for attr in group.get("attributes", [])
                    )

            rows.append({
                "id": item.get("id"),

                "name": item.get("name"),

                "lat": item.get("point", {}).get("lat"),
                "lon": item.get("point", {}).get("lon"),

                "district": district,

                "capacity": item.get("capacity", {}).get("total"),

                "rating": item.get("reviews", {}).get("general_rating"),

                "review_count":
                    item.get("reviews", {}).get("general_review_count"),

                "is_paid": item.get("is_paid"),

                "for_trucks": item.get("for_trucks"),

                "is_24x7":
                    item.get("schedule", {}).get("is_24x7"),

                "rubrics": rubrics,

                "payment_methods":
                    "; ".join(payment_methods),

                "full_name":
                    item.get("full_name"),

                "postcode":
                    item.get("address", {}).get("postcode"),

                "updated_at":
                    item.get("dates", {}).get("updated_at")
            })

        page += 1

        time.sleep(1)

    except Exception as e:
        print("ERROR:", e)
        break

df = pd.DataFrame(rows)

print("\nCollected:", len(df))

df.to_csv(
    "almaty_parking.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved: almaty_parking.csv")

print(df.head())


Page 1
Status: 200

NO RESULT FOUND
{
  "meta": {
    "code": 403,
    "error": {
      "message": "Authorization error (key is blocked, please contact api@2gis.ru)",
      "type": "apiKeyIsBlocked"
    },
    "api_version": "2.0.3068868",
    "issue_date": "20190201"
  }
}

Collected: 0

Saved: almaty_parking.csv
Empty DataFrame
Columns: []
Index: []


In [14]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

import pandas as pd
import time
import re


SEARCH_URL = "https://2gis.kz/almaty/search/парковки"


# -------------------
# DRIVER
# -------------------

options = Options()

options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options)

driver.get(SEARCH_URL)

time.sleep(8)

# -------------------
# SCROLL RESULTS
# -------------------

print("Collecting links...")

last_count = 0

for _ in range(50):

    driver.execute_script(
        """
        let panel=document.querySelector('[class*="scroll"]');
        if(panel){panel.scrollTop=panel.scrollHeight;}
        """
    )

    time.sleep(2)

    links = driver.find_elements(
        By.XPATH,
        "//a[contains(@href,'/firm/') or contains(@href,'/geo/')]"
    )

    count = len(links)

    print("Found:", count)

    if count == last_count:
        break

    last_count = count


# -------------------
# UNIQUE URLS
# -------------------

urls = set()

for a in links:

    try:
        href = a.get_attribute("href")

        if href:
            urls.add(href)

    except:
        pass


print("Total URLs:", len(urls))


# -------------------
# PARSE EACH CARD
# -------------------

rows = []

for i, url in enumerate(urls, start=1):

    try:

        print(i, "/", len(urls))

        driver.get(url)

        time.sleep(3)

        page = driver.page_source

        # -------------------
        # NAME
        # -------------------

        name = None

        try:
            name = driver.find_element(
                By.TAG_NAME,
                "h1"
            ).text
        except:
            pass

        # -------------------
        # ADDRESS
        # -------------------

        address = None

        try:

            elems = driver.find_elements(
                By.XPATH,
                "//*[contains(text(),'Алматы')]"
            )

            if elems:
                address = elems[0].text

        except:
            pass

        # -------------------
        # RATING
        # -------------------

        rating = None

        match = re.search(r'([0-5]\.[0-9])', page)

        if match:
            rating = match.group(1)

        # -------------------
        # CAPACITY
        # -------------------

        capacity = None

        cap_match = re.search(
            r'Вместимость.*?(\d+)',
            page,
            re.S
        )

        if cap_match:
            capacity = cap_match.group(1)

        # -------------------
        # COORDINATES
        # -------------------

        lat = None
        lon = None

        coord_match = re.search(
            r'@([0-9\.]+),([0-9\.]+)',
            driver.current_url
        )

        if coord_match:

            lon = coord_match.group(1)
            lat = coord_match.group(2)

        rows.append({

            "name": name,
            "address": address,
            "capacity": capacity,
            "rating": rating,
            "lat": lat,
            "lon": lon,
            "url": url

        })

    except Exception as e:

        print("ERROR:", e)


driver.quit()

# -------------------
# SAVE
# -------------------

df = pd.DataFrame(rows)

df.drop_duplicates(
    subset=["url"],
    inplace=True
)

df.to_csv(
    "almaty_parking.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df.shape)
print("DONE")

Found: 12
Found: 12
Total URLs: 12
1 / 12
2 / 12
3 / 12
4 / 12
5 / 12
6 / 12
7 / 12
8 / 12
9 / 12
10 / 12
11 / 12
12 / 12
(12, 7)
DONE


In [24]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

import pandas as pd
import re
import time


# ==========================
# CONFIG
# ==========================

SEARCH_QUERY = "парковка"
MAX_PAGES = 20

OUTPUT_FILE = "almaty_parking.csv"

# ==========================
# DRIVER
# ==========================

options = Options()

# Uncomment for headless mode
# options.add_argument("--headless=new")

options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

# ==========================
# STAGE 1
# COLLECT URLS
# ==========================

urls = set()

for page in range(1, MAX_PAGES + 1):

    url = f"https://2gis.kz/almaty/search/{SEARCH_QUERY}/page/{page}"

    print(f"\nPAGE {page}")

    driver.get(url)

    time.sleep(5)

    cards = driver.find_elements(
        By.CSS_SELECTOR,
        'a[href*="/firm/"]'
    )

    before = len(urls)

    for card in cards:

        href = card.get_attribute("href")

        if href:
            urls.add(href)

    print("New:", len(urls) - before)
    print("Total:", len(urls))

    # stop if no results
    if len(cards) == 0:
        break


print("\nTOTAL URLS:", len(urls))


# ==========================
# HELPERS
# ==========================

def find_capacity(text):

    patterns = [
        r"Мест всего\s*(\d+)",
        r"Вместимость.*?(\d+)",
        r"Количество мест.*?(\d+)"
    ]

    for p in patterns:

        m = re.search(
            p,
            text,
            flags=re.I | re.S
        )

        if m:
            return m.group(1)

    return None


def extract_rating(text):

    m = re.search(r"([1-5]\.\d)", text)

    if m:
        return m.group(1)

    return None


def has_word(text, word):

    return word.lower() in text.lower()

# ==========================
# STAGE 2
# OPEN EACH CARD
# ==========================

rows = []

urls = list(urls)

for idx, url in enumerate(urls, start=1):

    try:

        print(f"[{idx}/{len(urls)}] {url}")

        driver.get(url)

        time.sleep(3)

        html = driver.page_source

        # ------------------
        # NAME
        # ------------------

        name = ""

        try:
            name = driver.find_element(
                By.TAG_NAME,
                "h1"
            ).text.strip()
        except:
            pass

        # ------------------
        # ADDRESS
        # ------------------

        address = ""

        try:
            body_text = driver.find_element(
                By.TAG_NAME,
                "body"
            ).text

            for line in body_text.split("\n"):

                if "Алматы" in line:
                    address = line
                    break

        except:
            pass

        # ------------------
        # COORDINATES
        # ------------------

        lat = None
        lon = None

        coord_match = re.search(
            r'POINT\((\d+\.\d+)\s+(\d+\.\d+)\)',
            html
        )

        if coord_match:

            lon = float(coord_match.group(1))
            lat = float(coord_match.group(2))

        # ------------------
        # CAPACITY
        # ------------------

        capacity = None

        cap_match = re.search(
            r'"capacity":\{"total":"(\d+)"\}',
            html
        )

        if cap_match:
            capacity = int(cap_match.group(1))

        # ------------------
        # PRICE
        # ------------------

        hour_price = None
        day_price = None

        m = re.search(
            r'"name":"(\d+)\s*тнг\./час"',
            html
        )

        if m:
            hour_price = int(m.group(1))

        m = re.search(
            r'"name":"(\d+)\s*тнг\./сутки"',
            html
        )

        if m:
            day_price = int(m.group(1))

        paid = (
            hour_price is not None
            or day_price is not None
        )

        # ------------------
        # FEATURES
        # ------------------

        guarded = (
            'car_parking_guarded_car_parking' in html
            or '"name":"Охраняемая"' in html
        )

        indoor = (
            '"name":"Крытая"' in html
        )

        warm = (
            '"name":"Тёплая"' in html
        )

        truck = (
            '"for_trucks":true' in html
        )

        public_access = (
            '"access":"public"' in html
        )

        is_24_7 = (
            'Круглосуточно' in html
        )

        # ------------------
        # PAYMENTS
        # ------------------

        cash = (
            'general_payment_type_cash' in html
            or 'Наличный расчёт' in html
        )

        card = (
            'general_payment_type_card' in html
            or 'Оплата картой' in html
        )

        qr = (
            'general_payment_type_qr' in html
            or '"QR"' in html
        )

        # ------------------
        # RATING
        # ------------------

        rating = None

        rating_match = re.search(
            r'"rating":([0-9.]+)',
            html
        )

        if rating_match:
            rating = float(
                rating_match.group(1)
            )

        rows.append({

            "name": name,
            "address": address,

            "lat": lat,
            "lon": lon,

            "capacity": capacity,

            "hour_price": hour_price,
            "day_price": day_price,

            "rating": rating,

            "paid": paid,

            "guarded": guarded,
            "indoor": indoor,
            "warm": warm,

            "truck_parking": truck,

            "cash_payment": cash,
            "card_payment": card,
            "qr_payment": qr,

            "public_access": public_access,

            "open_24_7": is_24_7,

            "url": url

        })

    except Exception as e:

        print("ERROR:", url)
        print(e)
        
        name = ""

        try:
            h1 = driver.find_element(By.TAG_NAME, "h1")
            name = h1.text.strip()
        except:
            pass

        # ------------------
        # ADDRESS
        # ------------------

        address = ""

        lines = page_text.split("\n")

        for line in lines:

            if "Алматы" in line:

                address = line

                break

        # ------------------
        # COORDINATES
        # ------------------

        lat = None
        lon = None

        html = driver.page_source

        coord_match = re.search(
            r'([0-9]{2}\.[0-9]{5,})[^0-9]+([0-9]{2}\.[0-9]{5,})',
            html
        )

        if coord_match:

            lon = coord_match.group(1)
            lat = coord_match.group(2)

        # ------------------
        # FEATURES
        # ------------------

        capacity = find_capacity(page_text)

        paid = has_word(page_text, "Платная")

        guarded = has_word(page_text, "Охраняемая")

        indoor = has_word(page_text, "Крытая")

        warm = has_word(page_text, "Тёплая")

        truck = has_word(page_text, "грузовик")

        cash = has_word(page_text, "Наличный расчёт")

        card = has_word(page_text, "Оплата картой")

        qr = has_word(page_text, "QR")

        public_access = has_word(
            page_text,
            "Общедоступная"
        )

        rating = extract_rating(page_text)

        is_24_7 = has_word(
            page_text,
            "Круглосуточно"
        )

        rows.append({

            "name": name,

            "address": address,

            "lat": lat,
            "lon": lon,

            "capacity": capacity,

            "rating": rating,

            "paid": paid,

            "guarded": guarded,

            "indoor": indoor,

            "warm": warm,

            "truck_parking": truck,

            "cash_payment": cash,

            "card_payment": card,

            "qr_payment": qr,

            "public_access": public_access,

            "open_24_7": is_24_7,

            "url": url

        })

    except Exception as e:

        print("ERROR:", e)


# ==========================
# SAVE
# ==========================

driver.quit()

df = pd.DataFrame(rows)

df.drop_duplicates(
    subset=["url"],
    inplace=True
)

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nDONE")
print(df.shape)

print(df.head())


PAGE 1
New: 12
Total: 12

PAGE 2
New: 12
Total: 24

PAGE 3
New: 11
Total: 35

PAGE 4
New: 12
Total: 47

PAGE 5
New: 12
Total: 59

PAGE 6
New: 12
Total: 71

PAGE 7
New: 12
Total: 83

PAGE 8
New: 12
Total: 95

PAGE 9
New: 12
Total: 107

PAGE 10
New: 12
Total: 119

PAGE 11
New: 12
Total: 131

PAGE 12
New: 12
Total: 143

PAGE 13
New: 12
Total: 155

PAGE 14
New: 12
Total: 167

PAGE 15
New: 12
Total: 179

PAGE 16
New: 12
Total: 191

PAGE 17
New: 12
Total: 203

PAGE 18
New: 12
Total: 215

PAGE 19
New: 12
Total: 227

PAGE 20
New: 12
Total: 239

TOTAL URLS: 239
[1/239] https://2gis.kz/almaty/firm/70000001094799026?stat=eyJwYXJlbnRUYWJJZCI6IjU2NTAzYWZiLTk5NDEtNDVjZS04YjkyLTc2ZDE2YWM2MTMxMyIsImZvcmtFdmVudE9yZGluYWwiOjIsInVpRWxlbWVudCI6eyJuYW1lIjoicGxhY2VDYXJkTWluaSIsIm93bmVyTmFtZSI6InNlYXJjaFJlc3VsdHNMaXN0IiwicG9zaXRpb24iOjh9LCJwbGFjZUl0ZW0iOnsiYWRzSWQiOiIxMjI5MTIxNzIyOTczNzE0MTc5IiwiZW50aXR5Ijp7ImlkIjoiNzAwMDAwMDEwOTQ3OTkwMjYiLCJ0eXBlIjoiYnJhbmNoIiwic2VnbWVudEluZm8iOnsiYmFzZUxvY2FsZSI6InJ1X0taI

In [16]:
len(rows)

235

In [17]:
import pandas as pd

df = pd.DataFrame(rows)

print(df.shape)

df.to_csv(
    "almaty_parking_partial.csv",
    index=False,
    encoding="utf-8-sig"
)

df.head()

(235, 17)


,name,address,lat,lon,capacity,rating,paid,guarded,indoor,warm,truck_parking,cash_payment,card_payment,qr_payment,public_access,open_24_7,url
0,Парковка ТРК АДК,"Бостандыкский район, Алматы, 050046",57.984375,98.05597,None,1.5,True,False,False,False,False,False,False,False,False,False,https://2gis.kz/almaty/firm/70000001054874402?...
1,Автостоянка,Алматы,57.984375,98.05597,None,None,False,False,False,False,False,False,False,False,False,False,https://2gis.kz/almaty/firm/70000001057443788?...
2,Parking,Алматы,57.984375,98.05597,None,None,False,False,False,False,False,False,False,False,False,False,https://2gis.kz/almaty/firm/70000001089994206?...
3,Парковка,"Алмалинский район, Алматы",11.6405965,11.6405965,5,None,True,False,False,False,False,False,False,False,True,True,https://2gis.kz/almaty/firm/70030076861608325?...
4,Парковка,"Алмалинский район, Алматы",11.6405965,11.6405965,5,None,True,False,False,False,False,False,False,False,True,True,https://2gis.kz/almaty/firm/70030076861608325?...


In [21]:
df = pd.DataFrame(rows)

df.head(20)

,name,address,lat,lon,capacity,rating,paid,guarded,indoor,warm,truck_parking,cash_payment,card_payment,qr_payment,public_access,open_24_7,url
0,Парковка ТРК АДК,"Бостандыкский район, Алматы, 050046",57.984375,98.05597,None,1.5,True,False,False,False,False,False,False,False,False,False,https://2gis.kz/almaty/firm/70000001054874402?...
1,Автостоянка,Алматы,57.984375,98.05597,None,None,False,False,False,False,False,False,False,False,False,False,https://2gis.kz/almaty/firm/70000001057443788?...
2,Parking,Алматы,57.984375,98.05597,None,None,False,False,False,False,False,False,False,False,False,False,https://2gis.kz/almaty/firm/70000001089994206?...
3,Парковка,"Алмалинский район, Алматы",11.6405965,11.6405965,5,None,True,False,False,False,False,False,False,False,True,True,https://2gis.kz/almaty/firm/70030076861608325?...
4,Парковка,"Алмалинский район, Алматы",11.6405965,11.6405965,5,None,True,False,False,False,False,False,False,False,True,True,https://2gis.kz/almaty/firm/70030076861608325?...
5,Parking,Алматы,57.984375,98.05597,None,None,False,False,False,False,False,False,False,False,False,False,https://2gis.kz/almaty/firm/70000001089994206?...
6,Автостоянка №26,Алматы,57.984375,98.05597,None,4.7,False,False,False,False,False,False,False,False,False,False,https://2gis.kz/almaty/firm/9429940000791465?s...
7,Парковка ТРЦ Ритц-Палас,Алматы,57.984375,98.05597,None,2.4,False,False,False,False,False,False,False,False,False,False,https://2gis.kz/almaty/firm/70000001054875981?...
8,Parking,Алматы,57.984375,98.05597,None,None,False,False,False,False,False,False,False,False,False,False,https://2gis.kz/almaty/firm/70000001089994206?...
9,Парковка БЦ Нурлы Тау,Алматы,57.984375,98.05597,None,1.7,False,False,False,False,False,False,False,False,False,False,https://2gis.kz/almaty/firm/70000001110906180?...


In [34]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

import pandas as pd
import re
import time


# ==========================
# CONFIG
# ==========================

SEARCH_QUERY = "парковка"
MAX_PAGES = 1

OUTPUT_FILE = "almaty_parking.csv"

# ==========================
# DRIVER
# ==========================

options = Options()

# Uncomment for headless mode
# options.add_argument("--headless=new")

options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

# ==========================
# STAGE 1
# COLLECT URLS
# ==========================

urls = set()

for page in range(1, MAX_PAGES + 1):

    url = f"https://2gis.kz/almaty/search/{SEARCH_QUERY}/page/{page}"

    print(f"\nPAGE {page}")

    driver.get(url)

    time.sleep(5)

    cards = driver.find_elements(
        By.CSS_SELECTOR,
        'a[href*="/firm/"]'
    )

    before = len(urls)

    for card in cards:

        href = card.get_attribute("href")

        if href:
            urls.add(href)

    print("New:", len(urls) - before)
    print("Total:", len(urls))

    # stop if no results
    if len(cards) == 0:
        break


print("\nTOTAL URLS:", len(urls))


# ==========================
# STAGE 2
# OPEN EACH CARD
# ==========================

rows = []

for idx, url in enumerate(urls, start=1):

    print(f"\n[{idx}/{len(urls)}]")

    try:

        driver.get(url)
        time.sleep(4)

        html = driver.page_source

        # ------------------
        # NAME
        # ------------------

        name = ""

        try:
            h1 = driver.find_element(By.TAG_NAME, "h1")
            name = h1.text.strip()
        except Exception:
            pass

        # ------------------
        # ADDRESS
        # ------------------

        address = ""

        try:
            address_match = re.search(
                r'"components":\[\{"comment":"([^"]+)"',
                html
            )

            if address_match:
                address = address_match.group(1)
        except Exception:
            pass

        # ------------------
        # COORDINATES
        # ------------------

        lat = None
        lon = None

        coord_match = re.search(
            r'POINT\((\d+\.\d+)\s+(\d+\.\d+)\)',
            html
        )

        if coord_match:
            lon = float(coord_match.group(1))
            lat = float(coord_match.group(2))

        # ------------------
        # CAPACITY
        # ------------------

        capacity = None

        cap_match = re.search(
            r'"capacity":\{"total":"(\d+)"\}',
            html
        )

        if cap_match:
            capacity = int(cap_match.group(1))

        # ------------------
        # PRICE
        # ------------------

        hour_price = None
        day_price = None

        m = re.search(
            r'"name":"(\d+)\s*тнг\./час"',
            html
        )

        if m:
            hour_price = int(m.group(1))

        m = re.search(
            r'"name":"(\d+)\s*тнг\./сутки"',
            html
        )

        if m:
            day_price = int(m.group(1))

        paid = (
            hour_price is not None
            or day_price is not None
        )

        # ------------------
        # FEATURES
        # ------------------

        guarded = (
            'car_parking_guarded_car_parking' in html
            or '"name":"Охраняемая"' in html
        )

        indoor = (
            '"name":"Крытая"' in html
        )

        warm = (
            '"name":"Тёплая"' in html
        )

        truck = (
            '"for_trucks":true' in html
        )

        public_access = (
            '"access":"public"' in html
        )

        is_24_7 = (
            'Круглосуточно' in html
        )

        # ------------------
        # PAYMENT METHODS
        # ------------------

        cash = (
            'general_payment_type_cash' in html
            or 'Наличный расчёт' in html
        )

        card = (
            'general_payment_type_card' in html
            or 'Оплата картой' in html
        )

        qr = (
            'general_payment_type_qr' in html
            or '"QR"' in html
        )

        # ------------------
        # RATING
        # ------------------

        rating = None

        rating_match = re.search(
            r'"rating":([0-9.]+)',
            html
        )

        if rating_match:
            rating = float(rating_match.group(1))

        # ------------------
        # SAVE
        # ------------------

        rows.append({

            "name": name,
            "address": address,

            "lat": lat,
            "lon": lon,

            "capacity": capacity,

            "hour_price": hour_price,
            "day_price": day_price,

            "rating": rating,

            "paid": paid,

            "guarded": guarded,
            "indoor": indoor,
            "warm": warm,

            "truck_parking": truck,

            "cash_payment": cash,
            "card_payment": card,
            "qr_payment": qr,

            "public_access": public_access,
            "open_24_7": is_24_7,

            "url": url

        })

    except Exception as e:
        print(f"Error on {url}: {e}")
        continue


# ==========================
# SAVE TO CSV
# ==========================

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"\nSaved {len(df)} rows to {OUTPUT_FILE}")


# ==========================
# CLEANUP
# ==========================

driver.quit()


PAGE 1
New: 12
Total: 12

TOTAL URLS: 12

[1/12]

[2/12]

[3/12]

[4/12]

[5/12]

[6/12]

[7/12]

[8/12]

[9/12]

[10/12]

[11/12]

[12/12]

Saved 12 rows to almaty_parking.csv


In [36]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

import pandas as pd
import re
import time


# ==========================
# CONFIG
# ==========================

SEARCH_QUERY = "парковка"
MAX_PAGES = 15

OUTPUT_FILE = "almaty_parking.csv"

# ==========================
# DRIVER
# ==========================

options = Options()

# Uncomment for headless mode
# options.add_argument("--headless=new")

options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

# ==========================
# STAGE 1
# COLLECT URLS
# ==========================

urls = set()

for page in range(1, MAX_PAGES + 1):

    url = f"https://2gis.kz/almaty/search/{SEARCH_QUERY}/page/{page}"

    print(f"\nPAGE {page}")

    driver.get(url)

    time.sleep(5)

    cards = driver.find_elements(
        By.CSS_SELECTOR,
        'a[href*="/firm/"]'
    )

    before = len(urls)

    for card in cards:

        href = card.get_attribute("href")

        if href:
            urls.add(href)

    print("New:", len(urls) - before)
    print("Total:", len(urls))

    # stop if no results
    if len(cards) == 0:
        break


print("\nTOTAL URLS:", len(urls))


# ==========================
# STAGE 2
# OPEN EACH CARD
# ==========================

rows = []

for idx, url in enumerate(urls, start=1):

    print(f"\n[{idx}/{len(urls)}]")

    try:

        driver.get(url)
        time.sleep(4)

        html = driver.page_source

        # ---- TEMPORARY DEBUG: dump page source for first item ----
        # Run once, inspect almaty_debug_page.html for the real
        # field names around the address, then delete this block.
        if idx == 1:
            with open("almaty_debug_page.html", "w", encoding="utf-8") as f:
                f.write(html)
        # -----------------------------------------------------------

        # ------------------
        # NAME
        # ------------------

        name = ""

        try:
            h1 = driver.find_element(By.TAG_NAME, "h1")
            name = h1.text.strip()
        except Exception:
            pass

        # ------------------
        # ADDRESS
        # ------------------

        address = ""

        try:
            # adm_div is an array of {"type": "...", "name": "..."} entries
            # e.g. [{"type":"city","name":"Алматы"},
            #       {"type":"street","name":"Розыбакиева"},
            #       {"type":"building","name":"100"}]
            adm_div_match = re.search(
                r'"adm_div":\[(.*?)\]',
                html,
                flags=re.S
            )

            street = ""
            building = ""

            if adm_div_match:
                adm_div_raw = adm_div_match.group(1)

                # split into individual {...} entries
                entries = re.findall(r'\{[^{}]*\}', adm_div_raw)

                for entry in entries:

                    type_match = re.search(r'"type":"([^"]+)"', entry)
                    name_match = re.search(r'"name":"([^"]+)"', entry)

                    if not (type_match and name_match):
                        continue

                    entry_type = type_match.group(1)
                    entry_name = name_match.group(1)

                    if entry_type == "street":
                        street = entry_name
                    elif entry_type in ("building", "house"):
                        building = entry_name

            parts = []

            if street:
                parts.append(street)

            if building:
                parts.append(building)

            address = ", ".join(parts)

            if not address:
                # fallback: top-level address_name (often street + number)
                addr_match = re.search(
                    r'"address_name":"([^"]+)"',
                    html
                )

                if addr_match:
                    address = addr_match.group(1)

            if not address:
                # last resort: comment field (often just the city)
                comment_match = re.search(
                    r'"components":\[\{"comment":"([^"]+)"',
                    html
                )

                if comment_match:
                    address = comment_match.group(1)
        except Exception:
            pass

        # ------------------
        # COORDINATES
        # ------------------

        lat = None
        lon = None

        coord_match = re.search(
            r'POINT\((\d+\.\d+)\s+(\d+\.\d+)\)',
            html
        )

        if coord_match:
            lon = float(coord_match.group(1))
            lat = float(coord_match.group(2))

        # ------------------
        # CAPACITY
        # ------------------

        capacity = None

        cap_match = re.search(
            r'"capacity":\{"total":"(\d+)"\}',
            html
        )

        if cap_match:
            capacity = int(cap_match.group(1))

        # ------------------
        # PRICE
        # ------------------

        hour_price = None
        day_price = None

        m = re.search(
            r'"name":"(\d+)\s*тнг\./час"',
            html
        )

        if m:
            hour_price = int(m.group(1))

        m = re.search(
            r'"name":"(\d+)\s*тнг\./сутки"',
            html
        )

        if m:
            day_price = int(m.group(1))

        paid = (
            hour_price is not None
            or day_price is not None
        )

        # ------------------
        # FEATURES
        # ------------------

        guarded = (
            'car_parking_guarded_car_parking' in html
            or '"name":"Охраняемая"' in html
        )

        indoor = (
            '"name":"Крытая"' in html
        )

        warm = (
            '"name":"Тёплая"' in html
        )

        truck = (
            '"for_trucks":true' in html
        )

        public_access = (
            '"access":"public"' in html
        )

        is_24_7 = (
            'Круглосуточно' in html
        )

        # ------------------
        # PAYMENT METHODS
        # ------------------

        cash = (
            'general_payment_type_cash' in html
            or 'Наличный расчёт' in html
        )

        card = (
            'general_payment_type_card' in html
            or 'Оплата картой' in html
        )

        qr = (
            'general_payment_type_qr' in html
            or '"QR"' in html
        )

        # ------------------
        # RATING
        # ------------------

        rating = None

        rating_match = re.search(
            r'"rating":([0-9.]+)',
            html
        )

        if rating_match:
            rating = float(rating_match.group(1))

        # ------------------
        # SAVE
        # ------------------

        rows.append({

            "name": name,
            "address": address,

            "lat": lat,
            "lon": lon,

            "capacity": capacity,

            "hour_price": hour_price,
            "day_price": day_price,

            "rating": rating,

            "paid": paid,

            "guarded": guarded,
            "indoor": indoor,
            "warm": warm,

            "truck_parking": truck,

            "cash_payment": cash,
            "card_payment": card,
            "qr_payment": qr,

            "public_access": public_access,
            "open_24_7": is_24_7,

            "url": url

        })

    except Exception as e:
        print(f"Error on {url}: {e}")
        continue


# ==========================
# SAVE TO CSV
# ==========================

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"\nSaved {len(df)} rows to {OUTPUT_FILE}")


# ==========================
# CLEANUP
# ==========================

driver.quit()


PAGE 1
New: 12
Total: 12

PAGE 2
New: 12
Total: 24

PAGE 3
New: 11
Total: 35

PAGE 4
New: 12
Total: 47

PAGE 5
New: 12
Total: 59

PAGE 6
New: 12
Total: 71

PAGE 7
New: 12
Total: 83

PAGE 8
New: 12
Total: 95

PAGE 9
New: 12
Total: 107

PAGE 10
New: 12
Total: 119

PAGE 11
New: 12
Total: 131

PAGE 12
New: 12
Total: 143

PAGE 13
New: 12
Total: 155

PAGE 14
New: 12
Total: 167

PAGE 15
New: 12
Total: 179

TOTAL URLS: 179

[1/179]

[2/179]

[3/179]

[4/179]

[5/179]

[6/179]

[7/179]

[8/179]

[9/179]

[10/179]

[11/179]

[12/179]

[13/179]

[14/179]

[15/179]

[16/179]

[17/179]

[18/179]

[19/179]

[20/179]

[21/179]

[22/179]

[23/179]

[24/179]

[25/179]

[26/179]

[27/179]

[28/179]

[29/179]

[30/179]

[31/179]

[32/179]

[33/179]

[34/179]

[35/179]

[36/179]

[37/179]

[38/179]

[39/179]

[40/179]

[41/179]

[42/179]

[43/179]

[44/179]

[45/179]

[46/179]

[47/179]

[48/179]

[49/179]

[50/179]

[51/179]

[52/179]

[53/179]

[54/179]

[55/179]

[56/179]

[57/179]

[58/179]

[59/179]

In [37]:
df.head(10)

,name,address,lat,lon,capacity,hour_price,day_price,rating,paid,guarded,indoor,warm,truck_parking,cash_payment,card_payment,qr_payment,public_access,open_24_7,url
0,Парковка ТРК АДК,"Керей-Жанибек хандар улица, 558/1",43.232309,76.877023,NaN,400.0,1500.0,1.0,True,False,False,False,False,True,False,False,True,False,https://2gis.kz/almaty/firm/70000001054874402?...
1,Автостоянка №26,"улица Толе би, 287г",43.244332,76.849427,400.0,NaN,400.0,4.0,True,True,False,False,False,True,False,False,True,False,https://2gis.kz/almaty/firm/9429940000791465?s...
2,Автостоянка №26,"улица Толе би, 287г",43.244332,76.849427,400.0,NaN,400.0,4.0,True,True,False,False,False,True,False,False,True,False,https://2gis.kz/almaty/firm/9429940000791465?s...
3,Парковка ТРК АДК,"Керей-Жанибек хандар улица, 558/1",43.232309,76.877023,NaN,400.0,1500.0,1.0,True,False,False,False,False,True,False,False,True,False,https://2gis.kz/almaty/firm/70000001054874402?...
4,Парковка,"Керей-Жанибек хандар улица, 558/1",43.257023,76.932588,NaN,NaN,NaN,0.0,False,False,False,False,False,False,False,False,True,True,https://2gis.kz/almaty/firm/70030076861608325?...
5,Горный,"улица Зеина Шашкина, 36/1",43.220757,76.938259,100.0,NaN,NaN,5.0,False,False,True,False,False,True,False,False,True,True,https://2gis.kz/almaty/firm/9429940000794505?s...
6,Almaty Parking,"Керей-Жанибек хандар улица, 558/1",43.249876,76.948222,NaN,100.0,NaN,1.0,True,False,False,False,False,False,False,True,True,False,https://2gis.kz/almaty/firm/70000001057443905?...
7,Сайран-2,"5-й микрорайон, 40Б",43.234149,76.865536,70.0,500.0,500.0,5.0,True,True,True,False,False,True,False,True,True,False,https://2gis.kz/almaty/firm/9429940000799990?s...
8,Parking,"Керей-Жанибек хандар улица, 558/1",43.218470,76.842083,80.0,100.0,NaN,0.0,True,True,False,False,False,False,False,True,True,False,https://2gis.kz/almaty/firm/70000001089994206?...
9,Парковка,"Керей-Жанибек хандар улица, 558/1",43.257023,76.932588,NaN,NaN,NaN,0.0,False,False,False,False,False,False,False,False,True,True,https://2gis.kz/almaty/firm/70030076861608325?...
